# 10 · Set 4 setup: download Qwen2.5-32B (4-bit) and Llama-3.1-8B to Google Drive

Set 4 (`Set_1__3-modality-size`) uses three sizes per role:

| Role | Small | Medium | Large |
|---|---|---|---|
| Retriever | Qwen2.5-3B (have) | Qwen2.5-7B (have) | **Qwen2.5-32B, 4-bit AWQ (~19 GB)** |
| Answerer | Llama-3.2-3B (have) | **Llama-3.1-8B (~16 GB)** | Qwen2.5-32B (same files as the retriever) |

This notebook downloads the two models in bold (about 35 GB). A CPU runtime is fine.

**Llama-3.1-8B is gated.** Before running: open https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, accept the licence, wait for the approval
email, and make sure your Hugging Face token is saved in Colab's Secrets (key icon) as `HF_TOKEN` with notebook access on.

In [ ]:
# ==== settings ====
PROJECT_DIR = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"
DOWNLOAD = {
    "Qwen2.5-32B-Instruct-AWQ": "Qwen/Qwen2.5-32B-Instruct-AWQ",      # ~19 GB, not gated
    "Llama-3.1-8B-Instruct":    "meta-llama/Llama-3.1-8B-Instruct",   # ~16 GB, gated (needs HF_TOKEN + accepted licence)
    "Llama-3.2-3B-Instruct":    "meta-llama/Llama-3.2-3B-Instruct",   # already in Drive from earlier; skipped if complete
}
ALLOW  = ["*.json", "*.safetensors", "tokenizer*", "*.model", "*.txt", "*.tiktoken", "*.jinja"]
IGNORE = ["original/*", "*.pth", "*.bin", "*.gguf"]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, glob, time, json
!pip install -q hf_transfer        # huggingface_hub is already in Colab; upgrading it only triggers version warnings
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    tok = tok.get("value") if isinstance(tok, dict) else tok
except Exception:
    tok = None
print("Hugging Face token found:", bool(tok))
from huggingface_hub import snapshot_download

MODEL_DIR = f"{PROJECT_DIR}/models"
os.makedirs(MODEL_DIR, exist_ok=True)
total, used, free = shutil.disk_usage("/content/drive")
print(f"Drive before: {used/1e9:.1f} GB used, {free/1e9:.1f} GB free")
def missing_files(p):
    """Files a model folder still lacks: config, tokenizer and every weight file named in its index."""
    need = ["config.json", "tokenizer.json"]
    idx = f"{p}/model.safetensors.index.json"
    if os.path.exists(idx):
        need += sorted(set(json.load(open(idx))["weight_map"].values()))
    elif not glob.glob(f"{p}/*.safetensors"):
        need.append("model.safetensors")
    return [f for f in need if not (os.path.exists(f"{p}/{f}") and os.path.getsize(f"{p}/{f}") > 0)]
def complete(p):
    return os.path.isdir(p) and not missing_files(p)

for name, repo in DOWNLOAD.items():
    dst = f"{MODEL_DIR}/{name}"
    if complete(dst):
        print(f"✓ {name} already in Drive"); continue
    if os.path.isdir(dst):
        print(f"{name}: incomplete in Drive, missing {missing_files(dst)} - downloading again")
    t = time.time()
    tmp = f"/content/_dl/{name}"
    print(f"downloading {repo} ...")
    try:
        snapshot_download(repo_id=repo, local_dir=tmp, allow_patterns=ALLOW, ignore_patterns=IGNORE, token=tok or None)
    except Exception as e:
        if "meta-llama" in repo:
            print(f"✗ {name}: could not download ({type(e).__name__}). Accept the licence at https://huggingface.co/{repo}, "
                  "wait for approval, check the HF_TOKEN secret, then run this cell again.")
            continue
        raise
    print(f"copying to {dst} ...")
    shutil.copytree(tmp, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".cache"))
    shutil.rmtree(tmp)
    print(f"✓ {name} in {time.time() - t:.0f}s")

In [ ]:
!du -sh "{MODEL_DIR}"/*
total, used, free = shutil.disk_usage("/content/drive")
print(f"Drive after: {used/1e9:.1f} GB used, {free/1e9:.1f} GB free")
missing = {n: missing_files(f"{MODEL_DIR}/{n}") for n in DOWNLOAD if not complete(f"{MODEL_DIR}/{n}")}
print("All Set 4 models are complete in Drive." if not missing else f"STILL INCOMPLETE: {missing}")
from google.colab import drive
drive.flush_and_unmount()          # waits until every file is really uploaded to Drive
print("Drive upload finished - safe to close this notebook. Next: 11_set4_sweep.ipynb on an A100.")